# SG Transit Liveability — Demo Publish Workbook

Run this whenever you want to refresh the live demo before showing someone.

**Steps:**
1. Start the local pipeline (runs in the background — this cell returns immediately).
2. Let it run for ~10–15 minutes (re-run the status cell as often as you like while you wait — bus scores need that warm-up time).
3. Merge this session's data into the GitHub `pipeline-data` branch.
4. Regenerate and push the live snapshot page to your portfolio site.

Run the cells top to bottom. Nothing here blocks for the full 10–15 minutes — you decide when to move on.

In [ ]:
import subprocess, sys, time
from pathlib import Path
import requests

REPO_ROOT = Path.cwd()
PYTHON = str(REPO_ROOT / ".venv" / "Scripts" / "python.exe")
API = "http://localhost:8000"
SITE_REPO = "https://github.com/hm-base/Goh-Hui-Min-site.git"

print("Repo:", REPO_ROOT)
print("Python:", PYTHON)

## Step 1 — Start the local pipeline
Launches `python main.py` in the background. Safe to re-run this cell later in the session if the process ever dies — it'll just start a fresh one.

In [ ]:
# IMPORTANT: launching a second `main.py` while one is already running does NOT
# fail fast — it runs a full duplicate startup (its own TaxiWorker/BusWorker
# threads double-polling the LTA API and writing to the same SQLite file at
# once) for 40+ seconds before anything goes wrong. Always check first.
pipeline_proc = None

try:
    r = requests.get(f"{API}/health", timeout=2)
    already_up = r.ok
except requests.exceptions.ConnectionError:
    already_up = False

if already_up:
    print("Pipeline already running — not starting a duplicate:", r.json())
    print("(pipeline_proc stays None; the optional stop-cell will skip it.)")
else:
    pipeline_proc = subprocess.Popen(
        [PYTHON, "main.py"],
        cwd=REPO_ROOT,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    print(f"Started pipeline (pid={pipeline_proc.pid}) — waiting for it to come up...")

    for _ in range(30):
        try:
            r = requests.get(f"{API}/health", timeout=2)
            if r.ok:
                print("Pipeline is up:", r.json())
                break
        except requests.exceptions.ConnectionError:
            pass
        time.sleep(2)
    else:
        print("Still not responding after 60s — check for errors, or just re-run this cell.")

## Step 2 — Let it warm up
Re-run the cell below as many times as you like while you wait. Bus scores typically need ~15–30 minutes to fully populate; taxi data is usable within a minute or two.

In [ ]:
health = requests.get(f"{API}/health", timeout=5).json()
rank = requests.get(f"{API}/rank", timeout=5).json()
top = sorted(rank, key=lambda r: r["score"], reverse=True)[:3]

print("Snapshots collected this session:", health["snapshots"])
print("Top 3 districts right now:")
for r in top:
    print(f"  {r['district']:<20} score={r['score']:.1f}  bus={r['bus_frequency_score']:.0f}")

## Step 3 — Merge local data into the `pipeline-data` branch
Publishes the last N minutes of local (dense, 60s-interval) data to GitHub, so it coexists with — and wins over — anything GitHub Actions collected in the same window. Safe to run more than once; it won't duplicate rows.

In [ ]:
MINUTES = 60  # how far back to publish from this session

result = subprocess.run(
    [PYTHON, "scripts/publish_local_snapshot.py", "--minutes", str(MINUTES)],
    cwd=REPO_ROOT, capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print("--- STDERR ---")
    print(result.stderr)

## Step 4 — Regenerate and push the live snapshot page
Bakes the current live numbers into a static page and pushes it to your portfolio site repo. GitHub Pages usually rebuilds within ~1 minute after this.

In [ ]:
result = subprocess.run(
    [PYTHON, "scripts/publish_static_page.py", "--site-repo", SITE_REPO],
    cwd=REPO_ROOT, capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print("--- STDERR ---")
    print(result.stderr)
else:
    print("\nLive at:")
    print("  https://hm-base.github.io/Goh-Hui-Min-site/#projects")
    print("  https://hm-base.github.io/Goh-Hui-Min-site/sg-transit/index.html")

## Optional — stop the local pipeline
Run this when you're done. Not required — you can also just close the terminal/kernel.

In [ ]:
if pipeline_proc is None:
    print("No pipeline process was started by this notebook (it was already running) — nothing to stop here.")
else:
    pipeline_proc.terminate()
    print("Stopped pipeline (pid=%d)" % pipeline_proc.pid)